# Step 0 — Go / No-Go check of the pretrained Bikol→English model

**UNC Translator · AI model design, Section 5** (`docs/AI_MODEL_DESIGN.md`)

**Question this notebook answers:** does the existing model `Helsinki-NLP/opus-mt-bcl-en` already understand *our* Bikol-Naga well enough to be worth fine-tuning?
The model was mostly trained on formal religious text (its official test set is JW300), so we test it on **our own sentences** before spending time on training.

**What it does**
1. Loads our dataset (`corpus.csv`, `sentences.csv`).
2. Picks **50 sentences** (25 short, 25 long) with a fixed random seed, so the result can be repeated.
3. Translates them with the pretrained model, **unchanged** (no training).
4. Computes automatic scores: **chrF**, **BLEU**, **word-overlap F1**, plus the model's **confidence**.
5. Makes a **rating sheet** (Excel) for a Bikol-Naga speaker on the team.
6. After the sheet is filled in, computes the human-rating results and the **go / no-go decision**.

**How to run (Google Colab)**
1. In Colab: *File → Upload notebook* → choose this file.
2. *Runtime → Change runtime type → T4 GPU* (CPU also works, just slower).
3. Run the cells from top to bottom (*Runtime → Run all* stops at the upload prompts).
4. When asked, upload `data/corpus.csv` and `data/sentences.csv` from the project folder.

Nothing here changes the project or the dataset.

## 1. Install libraries

In [ ]:
!pip -q install transformers sentencepiece sacremoses sacrebleu openpyxl

## 2. Upload the dataset files

In [ ]:
from pathlib import Path

DATA = Path('data'); DATA.mkdir(exist_ok=True)
NEED = ['corpus.csv', 'sentences.csv']
missing = [f for f in NEED if not (DATA / f).exists()]
if missing:
    try:
        from google.colab import files
        print('Upload these files from the project data/ folder:', missing)
        for name, content in files.upload().items():
            clean_name = name.split(' (')[0] + ('.csv' if ' (' in name else '')   # Colab renames re-uploads to "corpus (1).csv"
            (DATA / clean_name).write_bytes(content)
    except ImportError:
        print('Not running on Colab: copy', missing, 'into', DATA.resolve())
print({f: (DATA / f).exists() for f in NEED})

## 3. Load and clean the sentence pairs

Uses the same cleaning as the system: `normalize()` from `engine.py` (to find duplicates) and `clean_english()` from `db.py` (removes notes in parentheses from the English).
Duplicates are dropped, keeping the first row, as the engine does.

In [ ]:
import csv, re, unicodedata

def clean(x):                                   # db.clean
    return re.sub(r'\s+', ' ', unicodedata.normalize('NFC', x or '')).strip()

def clean_english(e):                           # db.clean_english
    e = clean(e)
    m = re.search(r'\s*\(([^()]*)\)\s*$', e)
    if not m or not e[:m.start()].strip(): return e
    inner = m.group(1).strip()
    if inner.endswith(('...', '…')) or len(inner.split()) < 4: return e[:m.start()].strip()
    return inner

def normalize(text):                            # engine.normalize
    s = unicodedata.normalize('NFC', text).lower().replace('’', "'").replace('‘', "'")
    s = re.sub(r"[^\w\s'\-]", ' ', s)
    s = re.sub(r"(?<![^\W_])-|-(?![^\W_])", ' ', s)
    return re.sub(r'\s+', ' ', s).strip()

pairs, seen = [], set()
for f in NEED:
    with open(DATA / f, encoding='utf-8') as fh:
        for r in csv.DictReader(fh):
            bik, eng = clean(r['bikol_sentence']), clean_english(r['english_sentence'])
            key = normalize(bik)
            if not key or not eng or key in seen: continue
            seen.add(key)
            pairs.append({'bikol': bik, 'english': eng, 'source': f, 'words': len(key.split())})
print(len(pairs), 'unique sentence pairs loaded')

## 4. Pick the 50 test sentences

* **25 short** (3–5 words) and **25 long** (6 or more words), chosen at random with a fixed `SEED` so anyone re-running this gets the same 50.
* Two-word phrases are skipped: they test vocabulary, not sentences.
* **Optional:** to use your own picks (e.g. typical campus sentences), paste them into `CUSTOM` exactly as they appear in the dataset.
* Keep the saved `step0_sample.csv`: these 50 sentences should later go into the **test set**, so the fine-tuned model can be compared with the original on the same sentences.

In [ ]:
import random

SEED = 42
N_SHORT, N_LONG = 25, 25
CUSTOM = [
    # 'Paste a Bikol sentence from the dataset here',
]

rng = random.Random(SEED)
if CUSTOM:
    by_key = {normalize(p['bikol']): p for p in pairs}
    not_found = [s for s in CUSTOM if normalize(s) not in by_key]
    if not_found: print('Not found in the dataset (check spelling):', not_found)
    sample = [dict(by_key[normalize(s)], length=f"{by_key[normalize(s)]['words']} words") for s in CUSTOM if normalize(s) in by_key]
else:
    short = [p for p in pairs if 3 <= p['words'] <= 5]
    long_ = [p for p in pairs if p['words'] >= 6]
    sample = ([dict(p, length='short (3-5 words)') for p in rng.sample(short, N_SHORT)] +
              [dict(p, length='long (6+ words)') for p in rng.sample(long_, N_LONG)])

print(len(sample), 'sentences selected. First five:')
for i, p in enumerate(sample[:5], 1):
    print(f"{i}. {p['bikol']}  =>  {p['english']}")

## 5. Load the pretrained model (about 300 MB download)

In [ ]:
import time, torch
from transformers import MarianMTModel, MarianTokenizer

MODEL = 'Helsinki-NLP/opus-mt-bcl-en'
device = 'cuda' if torch.cuda.is_available() else 'cpu'
tok = MarianTokenizer.from_pretrained(MODEL)
model = MarianMTModel.from_pretrained(MODEL).to(device).eval()
print('device:', device, '| parameters:', f'{sum(p.numel() for p in model.parameters()) / 1e6:.1f} million')

## 6. Translate the 50 sentences

* **Beam search** with 4 beams (Section 7.4 of the design).
* **Confidence** = the model's average probability per output word piece (from the beam score), between 0 and 1. Section 7.7 plans to use it as a guardrail; this run shows whether it actually separates good from bad outputs (see step 10).

In [ ]:
def translate(texts, beams=4, batch=16):
    results = []
    for i in range(0, len(texts), batch):
        enc = tok(texts[i:i + batch], return_tensors='pt', padding=True, truncation=True, max_length=128).to(device)
        with torch.no_grad():
            out = model.generate(**enc, num_beams=beams, max_new_tokens=128,
                                 return_dict_in_generate=True, output_scores=True)
        texts_out = tok.batch_decode(out.sequences, skip_special_tokens=True)
        confidence = out.sequences_scores.exp().tolist()   # length-normalized log-probability -> probability
        results += list(zip(texts_out, confidence))
    return results

t0 = time.perf_counter()
res = translate([p['bikol'] for p in sample])
elapsed = time.perf_counter() - t0
for p, (eng, conf) in zip(sample, res):
    p['model_english'], p['confidence'] = eng, round(conf, 3)
print(f'{len(sample)} sentences translated in {elapsed:.1f} s on {device}')

## 7. Automatic scores

* **chrF** — character-level match with our English (0–100). Main automatic metric in the design.
* **BLEU** — word n-gram match (0–100). Reported because readers expect it; harsh on short sentences.
* **Word-overlap F1** — share of words in common (0–1), the same idea as the measure in manuscript Table 4.4. *Check that it matches the exact method used for that table before comparing numbers.*

These only measure similarity to the dataset's wording. A correct translation with different words scores low — that is why the human rating (step 9) decides.

In [ ]:
import sacrebleu
from collections import Counter

def words(s): return re.findall(r"[a-z0-9']+", s.lower())

def overlap_f1(hyp, ref):
    h, r = Counter(words(hyp)), Counter(words(ref))
    common = sum((h & r).values())
    if not common: return 0.0
    precision, recall = common / sum(h.values()), common / sum(r.values())
    return 2 * precision * recall / (precision + recall)

for p in sample:
    p['chrf'] = round(sacrebleu.sentence_chrf(p['model_english'], [p['english']]).score, 1)
    p['f1'] = round(overlap_f1(p['model_english'], p['english']), 2)

def report(group, name):
    hyps, refs = [p['model_english'] for p in group], [[p['english'] for p in group]]
    print(f"{name:<20} n={len(group):<3} chrF={sacrebleu.corpus_chrf(hyps, refs).score:5.1f}  "
          f"BLEU={sacrebleu.corpus_bleu(hyps, refs).score:5.1f}  "
          f"F1={sum(p['f1'] for p in group) / len(group):.2f}  "
          f"confidence={sum(p['confidence'] for p in group) / len(group):.2f}")

report(sample, 'All')
for g in sorted({p['length'] for p in sample}):
    report([p for p in sample if p['length'] == g], g)

## 8. Look at the outputs

In [ ]:
import pandas as pd
pd.set_option('display.max_colwidth', 80)
view = pd.DataFrame(sample)[['length', 'bikol', 'english', 'model_english', 'confidence', 'chrf']]
view.index = range(1, len(view) + 1)
view

### Optional: speed on CPU

The server will run the model on a **CPU**, not a GPU. This times one sentence at a time on Colab's CPU — only a rough guide; repeat the measurement on the real server later (requirement: ≤ 3 seconds).

In [ ]:
model.to('cpu'); device = 'cpu'
times = []
for p in sample[:20]:
    t = time.perf_counter(); translate([p['bikol']]); times.append(time.perf_counter() - t)
print(f'CPU, one sentence at a time: average {sum(times) / len(times):.2f} s, max {max(times):.2f} s')

## 9. Make the rating sheet for a Bikol-Naga speaker

Downloads **`step0_rating_sheet.xlsx`** (with drop-down lists) and **`step0_sample.csv`**.

The rater fills in two columns for every row:
* **style** — `casual` (everyday / campus talk) or `formal`.
* **rating** — judge the *meaning of the Bikol*, not just whether the words match the dataset's English:

| Rating | Meaning |
|---|---|
| **correct** | Same meaning as the Bikol, natural English. Wording may differ from the dataset. |
| **acceptable** | Meaning is understandable; small grammar or word-choice errors. |
| **incorrect** | Wrong meaning, important part missing, or nonsense. |

*These definitions are a proposal — confirm them with the adviser (manuscript Ch. 3 says the definitions are set with the adviser).*

In [ ]:
from openpyxl.styles import Alignment, Font
from openpyxl.worksheet.datavalidation import DataValidation

sheet = pd.DataFrame([{
    'no': i, 'length': p['length'], 'bikol': p['bikol'],
    'reference_english (dataset)': p['english'], 'model_english': p['model_english'],
    'confidence': p['confidence'], 'chrf': p['chrf'], 'f1': p['f1'],
    'style': '', 'rating': '', 'notes': ''} for i, p in enumerate(sample, 1)])

OUT = 'step0_rating_sheet.xlsx'
with pd.ExcelWriter(OUT, engine='openpyxl') as xw:
    sheet.to_excel(xw, index=False, sheet_name='ratings')
    ws = xw.sheets['ratings']
    last = len(sheet) + 1
    for col, options in [('I', 'casual,formal'), ('J', 'correct,acceptable,incorrect')]:
        dv = DataValidation(type='list', formula1=f'"{options}"', allow_blank=True)
        ws.add_data_validation(dv); dv.add(f'{col}2:{col}{last}')
    for col, width in zip('ABCDEFGHIJK', [5, 16, 40, 40, 40, 11, 7, 6, 10, 12, 30]):
        ws.column_dimensions[col].width = width
    for row in ws.iter_rows(min_row=1, max_row=last):
        for c in row: c.alignment = Alignment(wrap_text=True, vertical='top')
    for c in ws[1]: c.font = Font(bold=True)
    ws.freeze_panes = 'C2'

    guide = xw.book.create_sheet('how to rate')
    for line in ['Fill in the "style" and "rating" columns for every row (use the drop-down lists).',
                 'style: casual = everyday / campus talk; formal = formal or written style.',
                 'rating: judge the MEANING of the Bikol sentence, not only the dataset wording.',
                 '  correct    = same meaning as the Bikol, natural English (wording may differ).',
                 '  acceptable = meaning understandable, small grammar or word-choice errors.',
                 '  incorrect  = wrong meaning, important part missing, or nonsense.',
                 'Use "notes" for anything unusual (e.g. the dataset English itself looks wrong).',
                 'Do not change the other columns. Save as .xlsx and upload it in step 10.']:
        guide.append([line])
    guide.column_dimensions['A'].width = 100

sheet.to_csv('step0_sample.csv', index=False)
print('Saved', OUT, 'and step0_sample.csv')
try:
    from google.colab import files
    files.download(OUT); files.download('step0_sample.csv')
except ImportError:
    pass

## 10. Results and decision (after the sheet is rated)

Upload the **filled-in** `step0_rating_sheet.xlsx`, then run this cell.

**Decision rule** (`GO_THRESHOLD` — agree on the number with the adviser before looking at the results):
* **GO** — at least `GO_THRESHOLD`% of the outputs are *correct* or *acceptable* → the model already understands our Bikol; fine-tuning it is the main plan (Option B).
* **NO-GO** — below the threshold → drop the pretrained model; the from-scratch options (A/C) become the main plan.

Fine-tuning usually improves a model a lot on its new domain, so the bar here is "understands the basics", not "already good enough to use".

In [ ]:
GO_THRESHOLD = 40   # percent of outputs rated correct or acceptable  (agree with the adviser first)
RATED = Path('step0_rating_sheet.xlsx')

try:
    from google.colab import files
    print('Upload the FILLED-IN rating sheet:')
    up = files.upload()
    RATED = Path(next(iter(up)))
except (ImportError, StopIteration):
    pass

r = pd.read_excel(RATED, sheet_name='ratings')
r['rating'] = r['rating'].astype(str).str.strip().str.lower()
r['style'] = r['style'].astype(str).str.strip().str.lower()
rated = r[r['rating'].isin(['correct', 'acceptable', 'incorrect'])]
print(f'{len(rated)} of {len(r)} rows rated\n')

def summary(df, name):
    n = len(df)
    if not n: return
    c = df['rating'].value_counts()
    ok = c.get('correct', 0) + c.get('acceptable', 0)
    print(f"{name:<22} n={n:<3} correct={c.get('correct', 0):<3} acceptable={c.get('acceptable', 0):<3} "
          f"incorrect={c.get('incorrect', 0):<3} -> {100 * ok / n:5.1f}% correct or acceptable")

summary(rated, 'All')
for g in sorted(rated['length'].unique()): summary(rated[rated['length'] == g], g)
for s in ['casual', 'formal']: summary(rated[rated['style'] == s], s)

print('\nAverage confidence by rating (is confidence useful as a guardrail?):')
print(rated.groupby('rating')['confidence'].agg(['count', 'mean']).round(3).to_string())

ok_pct = 100 * rated['rating'].isin(['correct', 'acceptable']).mean() if len(rated) else 0
print(f"\nDECISION: {'GO - fine-tune this model (Option B)' if ok_pct >= GO_THRESHOLD else 'NO-GO - use the from-scratch plan (Options A/C)'}"
      f"  ({ok_pct:.1f}% vs threshold {GO_THRESHOLD}%)")

## 11. What to keep for the manuscript

Save these in the project under `ai/step0/`:
* `step0_sample.csv` — the 50 sentences and the model's outputs.
* the filled-in `step0_rating_sheet.xlsx`.
* the printed scores from steps 7 and 10 (copy them or take a screenshot), plus the date, `SEED`, and the rater (as a code, e.g. R-1, not a name).

They become the **Step 0 table in Chapter 4**: automatic scores (chrF, BLEU, F1), human ratings (correct / acceptable / incorrect, by length and style), and the decision.